In [ ]:
import pandas as pd

df_marking = pd.read_csv('df_with_marking_final.csv')

In [12]:
df_marking.loc[53, 'prison_term'] = 8.0
df_marking.loc[65, 'prison_term'] = 11.0
df_marking.loc[92, 'prison_term'] = 9.0

In [ ]:
import pandas as pd
import re
from pymorphy2 import MorphAnalyzer

morph = MorphAnalyzer()

NUM_WORDS = {
    'ноль': 0, 'один': 1, 'два': 2, 'три': 3, 'четыре': 4,
    'пять': 5, 'шесть': 6, 'семь': 7, 'восемь': 8, 'девять': 9,
    'десять': 10, 'одиннадцать': 11, 'двенадцать': 12, 'тринадцать': 13,
    'четырнадцать': 14, 'пятнадцать': 15, 'шестнадцать': 16,
    'семнадцать': 17, 'восемнадцать': 18, 'девятнадцать': 19,
    'двадцать': 20, 'тридцать': 30, 'сорок': 40, 'пятьдесят': 50,
    'шестьдесят': 60, 'семьдесят': 70, 'восемьдесят': 80, 'девяносто': 90
}

def lemmatize(word):
    if not word:
        return ''
    return morph.parse(word)[0].normal_form

def text2num(text):
    text = lemmatize(text)
    return NUM_WORDS.get(text.strip().lower(), 0)

def extract_number(raw):
    if not raw:
        return 0
    raw = raw.strip().lower()
    digit_match = re.search(r"\d+", raw)
    if digit_match:
        # return int(digit_match.group())
        return int(digit_match.group().lstrip("0") or "0")
    word_match = re.search(r"[а-я]+", raw)
    if word_match:
        return text2num(word_match.group())
    return 0

def get_full_text(row):
    """Объединяет все тексты дела в один"""
    return f"{str(row['sentence'])}"

prison_term_patterns = [
    r"окончательно назначить .*? наказание в виде ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) лет(?:\s*и?\s*((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) месяц[а-я]*)?",
    r"по совокупности.*?определить\s+[А-ЯЁ][а-яё]+\s+[А-ЯЁ]\.\s*[А-ЯЁ]\.\s*наказание в виде лишения свободы сроком на ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) (?:лет|года|год)(?:\s*([а-я\d\(\)\s]+?) месяц[а-я]*)?",
    r"по совокупности.*?назначить.*?наказание в виде лишения свободы сроком на ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) (?:лет|года|год)",
    r"путем частичного сложения назначенных наказаний, определить [а-яё]+\s+[а-яё]\.\s*[а-яё]\. наказание в виде ((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) (?:лет|года|год)(?:\s*((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) месяц[а-яё]*)? лишения свободы",
    r"назначить .*? наказание в виде лишения свободы на срок ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) (?:лет|года|год)(?:\s*и?\s*((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) месяц[а-я]*)?",
    r"назначить .*? наказание в виде лишения свободы сроком на ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) (?:лет|года|год)(?:\s*и?\s*((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) месяц[а-я]*)?",
    r"назначив .*? наказание в виде ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) лет(?:\s*и?\s*((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) месяц[а-я]*)? лишения свободы",
    r"назначить (?:ему|ей) наказание в виде ((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) (?:лет|года|год)(?:\s*((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) месяц[а-яё]*)? лишения свободы",
    r"назначить наказание в виде лишения свободы сроком на ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) лет(?:\s*((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) месяц[а-я]*)?",
    r"в виде лишения свободы сроком на ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) (?:лет|года|год)(?:\s*((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) месяц[а-я]*)?",
    r"назначить (?:ему|ей) наказание в виде лишения свободы,? сроком на ((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) (?:лет|года|год)(?:\s*((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) месяц[а-яё]*)?",
    r"назначить наказание в виде лишения свободы сроком ((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) (?:лет|года|год)(?:\s*((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) месяц[а-яё]*)?",
    r"назначить (?:ему|ей) наказание в виде лишения свободы сроком на ((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) (?:лет|года|год)",
    r"назначить (?:ему|ей)?\s*наказание\s+([а-яё]+)\s+(?:лет|года|год)\s+лишения свободы",
    r"назначить наказание в виде ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-яё]+)) (?:лет|года|год) лишения свободы",
    r"наказание в виде ограничения свободы на срок ((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) (?:год|года|лет)(?:\s*((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) месяц[а-яё]*)?",
    r"наказание .*? ((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) (?:лет|года|год)\s*((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) месяц[а-яё]* лишения свободы",
    r"лишения свободы на срок ((?:\d+|[а-яё]+)) (?:лет|года|год) ((?:\d+|[а-яё]+)) месяц[а-яё]*",
    r"наказание в виде лишения свободы на срок (\d+)лет\s*(\d+)?\s*месяц[а-я]*",
    r"назначить (?:ему|ей)?\s*наказание\s*(\d+)\s*\([а-я]+\)\s*лет\s*(\d+)?\s*месяц[а-я]*",
    r"наказание в виде\s*(\d+)\s*(?:лет|года|год)\s*лишения свободы",
    r"наказание в виде лишения свободы на срок\s*((?:\d+\s*\([а-яёё\s]+\)))",
    r"в виде ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) лет лишения свободы",
    r"назначить.*?наказание в виде\s*((?:\d+\s*/[а-яё]+/|[а-яё]+\s*/\d+/|\d+|[а-яё]+))\s*(?:лет|года|год)\s+лишения свободы"
]

train_data = []

for idx, row in df_marking.iterrows():
    text = get_full_text(row)
    text_lower = text.lower()
    text_flat = text_lower.replace("\n", " ").replace("\r", " ")
    target_term = row["prison_term"]

    try:
        target_term = float(target_term)
    except:
        continue

    if pd.isnull(target_term) or target_term == 0:
        continue

    matched = False

    if "пожизн" in text_flat:
        if abs(target_term - 100.0) < 0.1:
            start = text_flat.find("пожизн")
            end = start + len("пожизненное") if "пожизненное" in text_flat else start + len("пожизн")
            train_data.append((text, {"entities": [(start, end, "PRISON_TERM")]}))
            matched = True
        continue

    for pattern in prison_term_patterns:
        for match in re.finditer(pattern, text_flat):
            span = match.span()
            candidate_phrase = match.group(0)

            years_raw = match.group(1)
            months_raw = match.group(2) if len(match.groups()) > 1 else None
            years = extract_number(years_raw)
            months = extract_number(months_raw) if months_raw else 0
            extracted_term = round(years + months / 12, 2)

            if abs(extracted_term - target_term) < 0.1:
                train_data.append((text, {"entities": [(span[0], span[1], "PRISON_TERM")]}))
                matched = True
                break
        if matched:
            break

    if not matched:
        print(f"[!] Не найдено в id={row['id']}, срок: {target_term}")

print(f"TRAIN_DATA готово: {len(train_data)} примеров")

TRAIN_DATA готово: 92 примеров


In [ ]:
import spacy
from spacy.training.example import Example
from spacy.util import minibatch, compounding
import random

nlp = spacy.blank("ru")
ner = nlp.add_pipe("ner")

ner.add_label("PRISON_TERM")

optimizer = nlp.begin_training()
random.seed(42)

examples = []
for text, annotations in train_data:
    doc = nlp.make_doc(text)
    example = Example.from_dict(doc, annotations)
    examples.append(example)

for i in range(30):
    random.shuffle(examples)
    batches = minibatch(examples, size=compounding(4.0, 32.0, 1.5))
    for batch in batches:
        nlp.update(batch, drop=0.5, sgd=optimizer)

In [ ]:
def extract_predicted_prison_term(text):
    doc = nlp(text)
    for ent in doc.ents:
        if ent.label_ == "PRISON_TERM":
            years = extract_number(ent.text)
            months = extract_number(ent.text.split("месяц")[0]) if "месяц" in ent.text else 0
            return round(years + months / 12, 2)
    return None

y_true = []
y_pred = []

for _, row in df_marking.iterrows():
    true_val = row["prison_term"]
    try:
        true_val = float(true_val)
    except:
        continue
    if pd.isnull(true_val) or true_val == 0:
        continue

    text = get_full_text(row)
    predicted_val = extract_predicted_prison_term(text.lower().replace("\n", " ").replace("\r", " "))

    if predicted_val is not None:
        y_true.append(true_val)
        y_pred.append(predicted_val)
        print(true_val, predicted_val)

# accuracy с допустимой погрешностью
correct = sum(abs(t - p) < 1 for t, p in zip(y_true, y_pred))
accuracy = correct / len(y_true)

print(f"Accuracy: {accuracy:.2%} ({correct} / {len(y_true)})")

6.5 6.5
8.08 1.0
3.0 3.0
9.25 9.75
13.0 3.25
20.0 20.0
1.0 1.0
9.0 9.0
6.92 6.5
7.5 7.58
18.0 18.0
9.0 9.0
9.83 9.75
6.0 6.0
13.83 171.17
7.0 7.0
9.0 9.0
4.0 4.0
9.0 0.0
9.0 9.0
2.5 69.33
7.0 7.0
8.0 8.0
9.5 9.75
7.5 7.58
10.0 10.0
5.0 5.0
3.0 1.0
6.5 6.5
9.0 3.25
8.0 0.0
9.0 9.0
7.5 1.08
7.0 7.0
9.5 9.75
6.5 3.25
7.0 7.0
11.0 1.0
8.0 8.0
8.75 8.67
8.0 8.0
7.5 7.58
9.5 0.0
8.5 8.67
8.67 8.67
7.0 0.0
7.5 7.0
8.0 8.0
7.0 10.0
9.5 9.75
9.0 9.0
9.0 9.0
8.0 8.0
7.0 7.0
8.0 1.0
9.0 9.0
9.0 0.0
8.0 8.0
7.0 7.0
11.0 11.0
20.0 2.0
1.67 1.08
18.5 19.5
9.5 9.75
9.0 9.0
6.0 6.0
11.0 11.0
6.6 6.5
11.5 11.0
8.0 0.0
8.0 8.0
7.0 7.0
7.0 0.0
12.0 12.0
5.0 64.0
8.5 8.67
6.5 6.5
6.5 0.0
9.91 9.75
4.83 69.33
6.0 0.0
5.5 5.42
8.5 8.67
9.5 9.75
9.0 1.08
6.0 6.0
7.5 7.58
4.0 4.0
12.0 12.0
8.0 8.0
8.0 8.0
9.5 9.75
Accuracy: 72.83% (67 / 92)


In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

print(f"MAE: {mae:.2f} лет")
print(f"R²: {r2:.2f}")

MAE: 0.36 лет
R²: 0.86
